In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".")) # 把本 notebook 所在目录加进搜索路径，这样就能导入同目录下的 training_utils.py 与 transformer.py

import numpy as np
indices = np.random.randint(0, 100, size=5) # [0, 100)
print(f"indices: {indices}")

print("位置下标：")
print(f"{indices[:, None] + np.arange(0, 10)}")

indices: [46 16  7  0 76]
位置下标：
[[46 47 48 49 50 51 52 53 54 55]
 [16 17 18 19 20 21 22 23 24 25]
 [ 7  8  9 10 11 12 13 14 15 16]
 [ 0  1  2  3  4  5  6  7  8  9]
 [76 77 78 79 80 81 82 83 84 85]]


In [5]:
import numpy as np
import torch

# Problem 16: 实现数据加载 (2 分)
def get_batch(x: np.ndarray,
              batch_size: int,
              context_length: int,
              device: str) -> tuple[torch.Tensor, torch.Tensor]:
    '''
    x: np.ndarray  一维整数数组，元素为 token ID
    batch_size: int  B，批大小
    context_length: int  L，每条句子的上下文长度
    device: str  设备字符串，如 'cpu' or 'cuda:0'
    returns (input_seq, target_seq), 形状为 (batch_size, context_length)
    '''
    # 随机采样 B 个起始下标：取值范围 [0, len(x) - L + 1)，保证从起始下标出发，连续取 L 个 token 不会越界
    indices = np.random.randint(0, len(x) - context_length + 1, size=batch_size) # 最大起始下标为 len(x) - L, 句子里最后一个元素的下标为 len(x) - L + L - 1 < len(x) 

    # indices[:, None] 将 indices 的形状从 (B,) 变成 (B, L), 与 (B, L) 进行广播相加后形状变为 (B, L)
    # 1 -> 1 2 3 L; 2 -> 2 3 4 L+1; m -> m m+1 m+2 m+L-1
    # m m m ...m + 0 1 2 ... L-1 = m m+1 m+2 m+L-1
    input_seq = x[indices[:, None] + np.arange(0, context_length)]   # (len(x),), (B, L) -> (B, L): 以扩展后的下标数组为索引，在 x 的第一个维度上取元素

    indices += 1 # target_seq 的起始下标全部加 1
    target_seq = x[indices[:, None] + np.arange(0, context_length)]  # (B, L)

    # 转成 torch 张量并放到指定设备上
    input_seq_tensor = torch.tensor(input_seq, device=device)
    target_seq_tensor = torch.tensor(target_seq, device=device)

    return (input_seq_tensor, target_seq_tensor)

In [6]:
import numpy as np

# 使用例子：用一段很短的 token 序列观察采样结果
x = np.arange(20)   # 造一条长度为 20 的 token 序列：0, 1, 2, ..., 19
print("token 序列 x =", x)

inputs, targets = get_batch(x, batch_size=3, context_length=5, device="cpu")

print("\ninput  形状:", tuple(inputs.shape))
print(inputs)
print("target 形状:", tuple(targets.shape))
print(targets)

# 关键检查：target 的每一行 = input 对应行整体右移一格（第 i 位预测第 i+1 个 token）
print("\ntarget 是否等于 input 右移一格:", bool((targets == inputs + 1).all()))

token 序列 x = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]

input  形状: (3, 5)
tensor([[ 7,  8,  9, 10, 11],
        [ 0,  1,  2,  3,  4],
        [14, 15, 16, 17, 18]])
target 形状: (3, 5)
tensor([[ 8,  9, 10, 11, 12],
        [ 1,  2,  3,  4,  5],
        [15, 16, 17, 18, 19]])

target 是否等于 input 右移一格: True


In [7]:
import os
import typing
import torch

from torch.nn import Module
from torch.optim import Optimizer

# Problem 17: 实现模型检查点保存与加载 (1 分)
def save_checkpoint(model: Module,
                    optimizer: Optimizer,
                    iteration: int,
                    out: str | os.PathLike | typing.BinaryIO | typing.IO[bytes]) -> None:
    '''
    model: torch.nn.Module  模型
    optimizer: torch.optim.Optimizer  优化器
    iteration: int  当前的迭代步数
    out: str | os.PathLike | typing.BinaryIO | typing.IO[bytes]  保存路径或文件对象
    '''
    # state_dict() 返回保存全部可学习权重的字典；优化器同理（AdamW 的 m / v 等状态也在里面）
    torch.save(
        {
            "model": model.state_dict(),      # 模型权重
            "optim": optimizer.state_dict(),  # 优化器状态
            "iter_num": iteration,            # 迭代步数：恢复学习率调度要用
        }
        , out)


def load_checkpoint(src: str | os.PathLike | typing.BinaryIO | typing.IO[bytes],
                    model: Module,
                    optimizer: Optimizer = None) -> int:
    '''
    src: str | os.PathLike | typing.BinaryIO | typing.IO[bytes]  检查点路径或文件对象
    model: torch.nn.Module
    optimizer: torch.optim.Optimizer
    returns: int，检查点中保存的迭代步数
    '''
    device = next(model.parameters()).device   # 加载到模型当前所在的设备上
    state_dic = torch.load(src, map_location=device, weights_only=True)
    model.load_state_dict(state_dic["model"])
    if optimizer is not None:
        optimizer.load_state_dict(state_dic["optim"])

    return state_dic["iter_num"]

# 这份代码写出来就是这么心安，偷一下懒，不跑具体例子了

In [8]:
import torch

from torch.nn import Module
from torch.optim import Optimizer

# 复用前面实现好的组件：2.1 的交叉熵损失、2.2.4 的梯度裁剪
from training_utils import cross_entropy, gradient_clipping

# Problem 18: 组装完整训练循环 (4 分)
#
# 结构化拆法：先把一次参数更新封装成"单步训练函数"，再在 for 循环里反复调用它。
# 这里实现单步部分；完整循环（学习率调度 + 验证 + 检查点 + 日志）见同目录下的 train_llm.py。
#
# 【关于日志】train_llm.py 里用的是 wandb（Weights & Biases）——一个在线实验看板：
#   wandb.init(project="cs336-assignment1")   ≈ 打开一个网页端的实验记录本
#   wandb.log({"train_loss": 1.23})           ≈ print("train_loss = 1.23")，外加把数值画成曲线存进看板
#   wandb.finish()                            ≈ 关闭记录本
# 也就是说 wandb.log 本质上就是"带远程曲线的 print"，不做可视化时直接换回 print 即可。

def train_step(model: Module,
               optimizer: Optimizer,
               x: torch.Tensor,
               y: torch.Tensor,
               lr: float,
               max_l2_norm: float = 1.0) -> float:
    '''
    执行一步训练，返回本步的损失值（标量）

    model: torch.nn.Module  语言模型
    optimizer: torch.optim.Optimizer  优化器
    x: (batch_size, context_length)  输入 token ID，需为 long 类型
    y: (batch_size, context_length)  目标 token ID，即 x 整体右移一格
    lr: float  本步使用的学习率（由余弦调度给出）
    max_l2_norm: float  M，梯度裁剪允许的最大 ℓ2 范数
    '''
    optimizer.param_groups[0]["lr"] = lr   # 余弦调度给出的学习率，必须在 step 之前写进去

    logits = model(x)                      # (B, m, vocab_size) 前向传播
    loss = cross_entropy(logits, y).mean() # 对所有位置取平均，得到一个标量损失

    optimizer.zero_grad()                               # 清空上一步的梯度
    loss.backward()                                     # 反向传播，计算梯度
    gradient_clipping(model.parameters(), max_l2_norm)  # 梯度裁剪
    optimizer.step()                                    # 更新参数

    return loss.item()

In [9]:
import numpy as np
import torch

# 复用前面实现好的组件：2.2.2 的优化器、2.2.3 的学习率调度、1.5 的模型
from training_utils import AdamW, learning_rate_schedule
from transformer import TransformerLM

# 使用例子：用超小的 TransformerLM 跑一步，确认单步训练函数能跑通
# （完整训练循环：学习率调度 + 验证 + 检查点 + 日志，见同目录下的 train_llm.py）
torch.manual_seed(0)
vocab_size, context_length = 256, 32
model = TransformerLM(vocab_size=vocab_size,
                      context_length=context_length,
                      d_model=64,
                      num_layers=2,
                      num_heads=4,
                      d_ff=128,
                      rope_theta=10000.0)

# 优化器用 2.2.2 实现的 AdamW（接口与 torch.optim.AdamW 一致）
optimizer = AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.01)

# 造一份有规律的 toy 数据：0,1,2,...,255 循环出现
data = np.tile(np.arange(vocab_size, dtype=np.int32), 80)

print(f"数据集(部分): {data[:10]}")
print("随机初始化时约为 ln(256) ≈ 5.55")

for i in range(50):
    x, y = get_batch(data, batch_size=4, context_length=context_length, device="cpu")
    loss = train_step(model, optimizer, x.long(), y.long(), lr=1e-3)
    if i % 5 == 0:
        print(f"第 {i} 步训练后的损失：{loss:.4f}")

数据集(部分): [0 1 2 3 4 5 6 7 8 9]
随机初始化时约为 ln(256) ≈ 5.55
第 0 步训练后的损失：6.0119
第 5 步训练后的损失：5.5259
第 10 步训练后的损失：4.9422
第 15 步训练后的损失：4.6492
第 20 步训练后的损失：3.8003
第 25 步训练后的损失：3.5852
第 30 步训练后的损失：3.0501
第 35 步训练后的损失：3.1446
第 40 步训练后的损失：2.6153
第 45 步训练后的损失：2.4075
